# 00 — RAG Fundamentals

**Repo:** ai-learning-notes | **Folder:** rag-and-retrieval  
**Built alongside:** local-rag-llm (github.com/RT91-data/local-rag-llm)

---

## What this notebook covers

- Why RAG exists — the three problems with pure LLMs for document Q&A
- The full RAG pipeline — each stage explained with illustrative code
- Chunking and why it matters more than most tutorials admit
- How retrieval actually works — dense vs sparse, and why hybrid wins
- Where naive RAG breaks and what fixes it

**This is an explanatory notebook.** Code cells illustrate concepts with minimal dependencies — not wired to the full stack.

---
## 1. The Problem: Why Not Just Give the LLM Everything?

Three fundamental problems stop you from just pasting your documents into an LLM prompt:

### Problem 1: Context Window Limits
LLMs have a maximum input size. GPT-4 has 128K tokens. Claude Sonnet has ~200K tokens.  
A 50-page PDF is roughly 25,000 tokens. A company's document library is millions of tokens.  
**You cannot fit it all in the prompt.**

### Problem 2: Lost in the Middle
Even when documents *do* fit, LLMs don't read context windows uniformly.  
Research shows accuracy is high for information at the **start** and **end** of the context — information buried in the middle gets ignored.  
Stuffing 50 pages into context does not mean the LLM reads all 50 pages.

### Problem 3: Hallucination Without Grounding
Without a specific document to cite, the LLM answers from training data.  
Training data has a cutoff date and contains errors.  
**The LLM will confidently answer even when it doesn't know.**

RAG solves all three: retrieve only the relevant chunks (solves 1 and 2), then force the LLM to answer only from those chunks (solves 3).

In [ ]:
# Illustrating the context window problem
# Rough token estimates (1 token ≈ 4 characters)

documents = {
    "50-page PDF": 25_000,
    "D365 FnO documentation (full)": 2_000_000,
    "Company email archive (1 year)": 50_000_000,
    "ERP transaction history (5 years)": 500_000_000,
}

model_context_windows = {
    "GPT-4o": 128_000,
    "Claude Sonnet 4": 200_000,
    "Claude Opus 4": 200_000,
}

print(f"{'Document':<45} {'Tokens':>15} {'Fits in Claude?':>15}")
print("-" * 80)
for doc, tokens in documents.items():
    fits = "✅ Yes" if tokens <= 200_000 else "❌ No"
    print(f"{doc:<45} {tokens:>15,} {fits:>15}")

---
## 2. The RAG Pipeline — Overview

RAG has two distinct phases:

```
INDEXING PHASE (runs once, offline)
  Documents → Load → Chunk → Embed → Store in Vector DB

QUERY PHASE (runs per question, online)
  Question → Embed → Retrieve → Rerank → Generate → Answer
```

The indexing phase builds the search index.  
The query phase uses it to find the right chunks, then asks the LLM to answer from those chunks only.

**Key insight:** The LLM never reads the full document. It reads only the 4-5 most relevant chunks — roughly 2,000 tokens instead of 25,000. This is the core efficiency gain.

---
## 3. Stage 1 — Document Loading

Before chunking, you need to extract text from the source format.  
For PDFs, the extraction method matters significantly:

| Tool | Handles tables? | Speed | Notes |
|---|---|---|---|
| PyPDFLoader | ❌ Poorly | Fast | Merges table rows, loses structure |
| pdfplumber | ✅ Well | Medium | Cell-by-cell geometric extraction |
| Unstructured | ✅ Best | Slow | Enterprise-grade, handles complex layouts |

In `local-rag-llm`, pdfplumber is used because it correctly extracts tables by reading their geometric structure, converting each cell to pipe-separated text that LLMs can parse:

```
[TABLE]
Vendor | Payment Term | Outstanding Balance
CONTOSO-001 | NET30 | SGD 45,000
FABRIKAM-002 | NET60 | SGD 12,500
[/TABLE]
```

Without this, a D365 vendor balance table becomes garbled text the LLM cannot read.

---
## 4. Stage 2 — Chunking

Chunking splits the document into pieces that fit in the retriever's context.  
Two parameters control this: `chunk_size` (maximum characters per chunk) and `chunk_overlap` (how many characters the next chunk shares with the previous one).

**Why overlap matters:** If the answer spans a chunk boundary — say the question is answered across sentences 8-12 and your chunk ends at sentence 10 — overlap ensures both chunks contain the boundary content. Without overlap, you'd miss half the answer.

In [ ]:
# Illustrating how chunking works with overlap

def simple_chunk(text, chunk_size=100, overlap=20):
    """Illustrative chunker — not production grade."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap   # step back by overlap amount
    return chunks

text = (
    "Zero Ambient Authority means an agent must never inherit the developer's full "
    "administrative privileges. Instead the system uses Just-In-Time token downscoping. "
    "When an agent writes a new script the sandbox receives fresh hyper-restricted "
    "credentials scoped to only the data sources required for that specific task. "
    "These tokens expire the exact moment the task concludes."
)

chunks = simple_chunk(text, chunk_size=120, overlap=30)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i}: ...{chunk}...")
    print()

In [ ]:
# The chunk_size tradeoff — this is a real engineering decision

print("CHUNK SIZE TRADEOFFS")
print("=" * 60)

configs = [
    (500,  100, "Too small",  "loses context, answers get split, high precision queries fail"),
    (1000, 200, "Default",    "good for most single-topic answers, may split multi-point lists"),
    (2000, 400, "Current",    "handles multi-point answers (7 pillars), more context per chunk"),
    (4000, 800, "Too large",  "less precise retrieval, CrossEncoder works harder, slower"),
]

print(f"{'Size':>6} {'Overlap':>8} {'Label':>10}   Note")
print("-" * 80)
for size, overlap, label, note in configs:
    print(f"{size:>6} {overlap:>8} {label:>10}   {note}")

print()
print("Key insight from local-rag-llm eval:")
print("With chunk_size=1000, 'What are the 7 pillars?' scored context_recall=0.286")
print("because pillars 3-6 were split across a boundary and lost.")
print("With chunk_size=2000, context_recall improved to 0.857.")
print("The remaining miss: Pillar 6 description still truncated at chunk boundary.")

---
## 5. Stage 3 — Embeddings

An embedding is a list of numbers (a vector) that represents the *semantic meaning* of text.  
Texts with similar meaning have vectors that point in similar directions.  
This is how the retriever finds relevant chunks without matching exact keywords.

**How it works internally (simplified):**
- Text is tokenised into subword units
- Each token passes through a transformer encoder
- The final layer's output is pooled into a single fixed-size vector
- nomic-embed-text produces 768-dimensional vectors

**What the dimensions mean:** Nothing individually — it's the *relative position* of vectors in 768-dimensional space that encodes meaning. You cannot inspect dimension 42 and say "this represents finance".

In [ ]:
import numpy as np

# Simulating what embeddings look like and how similarity is measured
# Real embeddings from nomic-embed-text would be 768-dimensional
# We use 4-dimensional vectors here to keep it readable

np.random.seed(42)

def fake_embed(text):
    """Not a real embedding model — just illustrates the shape."""
    # In reality: send text to OllamaEmbeddings(model='nomic-embed-text')
    np.random.seed(hash(text) % 2**31)
    return np.random.randn(4)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Our question
question = "What is Zero Ambient Authority?"

# Candidate chunks from the document
chunks = {
    "ZAA chunk":     "Zero Ambient Authority means an agent must never inherit full admin privileges.",
    "Sandboxing":    "Ephemeral sandboxes reset state between runs to contain blast radius.",
    "Red team":      "The Red Team injects adversarial vibes to test agent robustness.",
    "Eval dims":     "Trajectory quality measures whether the agent took a sensible path.",
}

q_vec = fake_embed(question)

print(f"Query: '{question}'\n")
print(f"{'Chunk':<15} {'Similarity':>12}  {'Retrieve?':>10}")
print("-" * 45)

results = []
for name, chunk in chunks.items():
    c_vec = fake_embed(chunk)
    sim = cosine_similarity(q_vec, c_vec)
    results.append((name, sim))

results.sort(key=lambda x: x[1], reverse=True)
for name, sim in results:
    retrieve = "✅ Yes" if sim > 0.5 else "❌ No"
    print(f"{name:<15} {sim:>12.4f}  {retrieve:>10}")

print("\nNote: Real nomic-embed-text vectors are 768-dimensional.")
print("This is illustrative — real similarity scores work the same way.")

---
## 6. Stage 4 — Vector Store and Retrieval

A vector store is a database optimised for finding vectors that are close to a query vector.  
FAISS (Facebook AI Similarity Search) is a local, in-memory vector store that uses approximate nearest neighbour (ANN) search.

**Two types of retrieval — and why you need both:**

| | Dense (FAISS) | Sparse (BM25) |
|---|---|---|  
| **Basis** | Semantic meaning (vectors) | Keyword frequency (TF-IDF) |
| **Finds** | "JIT downscoping" when user asks "temporary credentials" | "SPIFFE ID" when user types "SPIFFE ID" exactly |
| **Misses** | Exact codes, IDs, product names | Paraphrased questions |
| **In local-rag-llm** | FAISS 60% weight | BM25 40% weight |

**Why hybrid retrieval wins:** A user asking "What is the permission model for agents?" gets semantic matches from FAISS. A user typing "SPIFFE ID" gets exact keyword matches from BM25. Neither retriever alone covers both cases.

In [ ]:
# Illustrating when dense retrieval fails and BM25 saves it

chunks = [
    "SPIFFE IDs are cryptographic identities assigned to every individual agent.",
    "Agents must never inherit broad administrative privileges from their parent.",
    "Trust Decay means trust is lost when sub-goals diverge from original intent.",
]

# BM25 simplified: score = number of query words found in chunk
def bm25_simple(query, chunk):
    query_words = set(query.lower().split())
    chunk_words = set(chunk.lower().split())
    return len(query_words & chunk_words)

queries = [
    "SPIFFE ID",                    # exact term — BM25 wins
    "how are agents identified?",   # semantic — dense wins
]

for query in queries:
    print(f"Query: '{query}'")
    print(f"{'Chunk (truncated)':<55} {'BM25':>5}")
    print("-" * 65)
    for chunk in chunks:
        score = bm25_simple(query, chunk)
        print(f"{chunk[:52]+'...':<55} {score:>5}")
    print()

print("Key insight:")
print("'SPIFFE ID' query → BM25 correctly ranks chunk 1 highest (exact match)")
print("Dense embeddings might rank chunk 2 higher (semantically related to identity)")
print("Hybrid retrieval combines both signals — you get the right chunk either way.")

---
## 7. Stage 5 — Reranking

After retrieval you have 16 candidate chunks. Not all of them are equally good.  
Reranking re-scores them using a more powerful but slower model — a **CrossEncoder**.

**Bi-encoder (what FAISS uses):**  
Question → [Encoder] → vector_Q  
Chunk → [Encoder] → vector_C  
Similarity = dot_product(vector_Q, vector_C)  
→ Fast. Question and chunk never "see" each other.

**CrossEncoder (what the reranker uses):**  
[Question + Chunk] → [Encoder] → single relevance score  
→ Slower. Model reads both together, so attention heads can relate question tokens to chunk tokens directly.

**Why not use CrossEncoder for retrieval directly?**  
You have 10,000 chunks. CrossEncoder would need 10,000 inference calls per query.  
FAISS narrows it to 16 candidates first. Then CrossEncoder runs 16 times. That's feasible.

In [ ]:
# Illustrating the computational cost difference

total_chunks = 500          # typical for a 50-page PDF at chunk_size=2000
faiss_candidates = 16       # what FAISS returns
crossencoder_top_k = 5      # what CrossEncoder keeps

# Inference time estimates (rough, on CPU)
faiss_search_ms = 5         # vector lookup is nearly instant
crossencoder_per_call_ms = 80  # one CrossEncoder forward pass on CPU

print("RETRIEVAL COST COMPARISON")
print("=" * 50)
print(f"Total chunks in index:         {total_chunks}")
print()

print("Option A: CrossEncoder over all chunks (naive)")
naive_ms = total_chunks * crossencoder_per_call_ms
print(f"  Calls: {total_chunks} × {crossencoder_per_call_ms}ms = {naive_ms:,}ms = {naive_ms/1000:.1f}s")
print("  Result: 40 seconds per query. Unusable.")
print()

print("Option B: FAISS → CrossEncoder (what local-rag-llm does)")
hybrid_ms = faiss_search_ms + (faiss_candidates * crossencoder_per_call_ms)
print(f"  FAISS: {faiss_search_ms}ms")
print(f"  CrossEncoder: {faiss_candidates} × {crossencoder_per_call_ms}ms = {faiss_candidates * crossencoder_per_call_ms}ms")
print(f"  Total: {hybrid_ms}ms = {hybrid_ms/1000:.2f}s")
print(f"  Speedup: {naive_ms // hybrid_ms}× faster, with better precision than FAISS alone")

---
## 8. Stage 6 — Generation

The final stage sends the retrieved chunks + the question to the LLM.

**The key constraint:** The system prompt must force the LLM to answer *only* from the provided context. Without this, the LLM will blend retrieved content with its training data — and you lose the grounding guarantee.

From `local-rag-llm`:

```python
system_prompt = """You are a precise and helpful document assistant.
Answer questions using ONLY the context provided.
If the answer is not in the context, say: 'I cannot find this in the provided documents.'
Always cite the source filename and page number for every fact you state.
Do not invent information not present in the context."""
```

**What the prompt does:**
- `ONLY the context provided` → prevents hallucination from training data  
- `I cannot find this` → explicit fallback prevents confident wrong answers  
- `cite the source` → forces traceability, makes hallucination visible  

**RAGAS Faithfulness** measures whether the answer honours this constraint — whether every claim in the output is actually supported by the retrieved context.

In [ ]:
# Illustrating what the full prompt looks like at query time

retrieved_chunks = [
    {
        "source": "Vibe_Coding_Security.pdf",
        "page": 19,
        "content": (
            "Zero Ambient Authority means an agent must never inherit the developer's full "
            "administrative privileges. Just-In-Time downscoping provides fresh, "
            "hyper-restricted credentials scoped to the exact data sources required."
        )
    },
    {
        "source": "Vibe_Coding_Security.pdf",
        "page": 11,
        "content": (
            "Access relies on Attribute-Based Access Control (ABAC) and Just-In-Time (JIT) "
            "token downscoping. This enforces a permissions matrix of Intent × User × Time, "
            "ensuring agents receive credentials that expire immediately after a task concludes."
        )
    },
]

question = "What is Zero Ambient Authority?"

# Build context string (what goes into the LLM prompt)
context_parts = []
for i, chunk in enumerate(retrieved_chunks, 1):
    context_parts.append(
        f"[Source {i}: {chunk['source']}, Page {chunk['page']}]\n{chunk['content']}"
    )
context = "\n\n".join(context_parts)

user_message = f"DOCUMENT CONTEXT:\n{context}\n\nQUESTION: {question}"

print("=" * 60)
print("WHAT THE LLM ACTUALLY RECEIVES")
print("=" * 60)
print()
print("[SYSTEM PROMPT]")
print("You are a precise document assistant. Answer ONLY from the")
print("context provided. Cite source and page for every fact.")
print()
print("[USER MESSAGE]")
print(user_message)
print()
print(f"Total characters sent to LLM: {len(user_message):,}")
print(f"Approx tokens: ~{len(user_message)//4:,}")

---
## 9. Where Naive RAG Breaks

Naive RAG = load → chunk → embed → retrieve top-k → generate.  
It works for simple factual questions. It fails in predictable ways:

| Failure Mode | Example | Root Cause |
|---|---|---|
| **Chunk boundary split** | "7 pillars" answer spans 3 pages | chunk_size too small |
| **TOC/header noise** | Table of Contents retrieved instead of content | No junk filter |
| **Cross-document contamination** | Wrong PDF's chunks retrieved | Mixed index, no source filter |
| **Vocabulary mismatch** | "temp credentials" doesn't match "JIT tokens" | Pure dense, no BM25 |
| **Conversational follow-up** | "What about the second one?" fails | No query rewriting |
| **Hallucination under pressure** | LLM answers anyway when context is empty | Weak system prompt |
| **Prompt injection** | Malicious text in PDF hijacks the LLM | No security layers |

Each of these failures has a specific fix.  
The next notebooks in this series cover each one.

In [ ]:
# Quantifying the failure modes observed during RAGAS evaluation
# of local-rag-llm with the 20-question golden dataset

eval_results = {
    # (question_id, topic, context_recall, root_cause_if_failed)
    "Q1":  ("7 pillars",          0.857, "Pillar 6 truncated at chunk boundary"),
    "Q2":  ("Confused Deputy",    1.000, None),
    "Q3":  ("Slopsquatting",      1.000, None),
    "Q4":  ("ZAA/JIT",            1.000, None),
    "Q5":  ("Vibe Diff",          1.000, None),
    "Q6":  ("7 eval dimensions",  1.000, None),
    "Q7":  ("DoW attack",         0.667, "Ground truth over-specified — fixed"),
    "Q8":  ("MCP Spoofing",       1.000, None),
    "Q9":  ("Intent Drift",       1.000, None),
    "Q10": ("AgBOM",              1.000, None),
    "Q11": ("Why eval different", 1.000, None),
    "Q12": ("Ephemeral sandbox",  1.000, None),
    "Q13": ("Effective Trust",    0.750, "Specific comparison sentence below threshold"),
    "Q14": ("Red/Blue/Green",     1.000, None),
    "Q15": ("App vulnerabilities",1.000, None),
    "Q16": ("Repo poisoning",     1.000, None),
    "Q17": ("Delegated identity", 1.000, None),
    "Q18": ("Session prefix",     1.000, None),
    "Q19": ("Green Team",         1.000, None),
    "Q20": ("Trajectory quality", 0.500, "Ground truth over-specified — fixed"),
}

passed = sum(1 for v in eval_results.values() if v[1] == 1.0)
failed = len(eval_results) - passed
avg_recall = sum(v[1] for v in eval_results.values()) / len(eval_results)

print(f"Context Recall Results — Clean Single-Document Baseline")
print(f"{'='*55}")
print(f"Questions:      {len(eval_results)}")
print(f"Perfect (1.0):  {passed}")
print(f"Imperfect:      {failed}")
print(f"Average recall: {avg_recall:.3f}")
print()
print("Failures and root causes:")
for qid, (topic, recall, cause) in eval_results.items():
    if recall < 1.0:
        print(f"  {qid} {topic:<25} recall={recall:.3f}  → {cause}")

---
## 10. Summary

| Stage | What it does | Key decision |
|---|---|---|
| **Load** | Extract text from source format | pdfplumber for table-aware extraction |
| **Chunk** | Split into retrievable pieces | chunk_size and overlap tradeoff |
| **Embed** | Convert text to vectors | nomic-embed-text (local, free) |
| **Store** | Index vectors for fast search | FAISS (local) or managed DB |
| **Retrieve** | Find relevant chunks | Hybrid: dense (60%) + sparse BM25 (40%) |
| **Rerank** | Re-score with CrossEncoder | Runs on top-16 candidates only |
| **Generate** | Answer from context only | System prompt enforces grounding |
| **Evaluate** | Measure quality | RAGAS: faithfulness, recall, precision, relevancy |

**The core insight of RAG:**  
LLMs are good at reasoning. They are bad at remembering.  
RAG gives the LLM a memory system — retrieval handles what to recall, the LLM handles what to conclude from it.

---
## Next Notebooks

- **01** — Document loading and chunking strategies (deep dive)
- **02** — Embeddings and vector stores (FAISS internals)
- **03** — Sparse retrieval and BM25
- **04** — Hybrid retrieval and fusion strategies
- **05** — Reranking with CrossEncoders
- **06** — RAG evaluation with RAGAS